<a href="https://colab.research.google.com/github/vikrampal12345/Machine_learning/blob/main/8_Column_Transformer_Feature_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# 1. Normal method

In [26]:
df = pd.read_csv("/content/covid_toy.csv")
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [27]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [28]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [29]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size = 0.2)

X_train.shape

(80, 5)

In [31]:
X_train

,age,gender,fever,cough,city
56,71,Male,NaN,Strong,Kolkata
81,65,Male,99.0,Mild,Delhi
98,5,Female,98.0,Strong,Mumbai
57,49,Female,99.0,Strong,Bangalore
75,5,Male,102.0,Mild,Kolkata
...,...,...,...,...,...
62,56,Female,104.0,Strong,Bangalore
52,47,Female,100.0,Strong,Bangalore
31,83,Male,103.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata


In [32]:
# adding simple imputer to fever col
# SimpleImputer() - Nan replace by means

si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [ ]:
X_train_fever

In [34]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']]) # it should be two dimension

X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [35]:
X_train.shape

(80, 5)

In [ ]:
ohe = OneHotEncoder(drop='first')

X_train_g_c = ohe.fit_transform(X_train[['gender', 'city']])

X_test_g_c = ohe.transform(X_test[['gender', 'city']])

X_train_g_c.toarray()

In [37]:
X_train_g_c.shape

(80, 4)

In [38]:
# Extract the age

X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [39]:
X_train_age.shape

(80, 1)

In [40]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_g_c, X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_g_c, X_test_cough),axis=1)

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 2 has 0 dimension(s)

In [ ]:
df.head(4)

# 2. Mentos Zindagi

In [44]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder = 'passthrough')

In [46]:
transformer.fit_transform(X_train).shape

(80, 7)

In [47]:
transformer.transform(X_test).shape

(20, 7)

# ColumnTransformer — Short Notes

## What is `ColumnTransformer`?

`ColumnTransformer` applies **different preprocessing techniques to different columns** in a dataset.

Example:

* Numerical columns → Scaling
* Categorical columns → Encoding
* Missing values → Imputation

---

## Syntax

```python
ColumnTransformer(
    transformers=[
        (name, transformer, columns)
    ],
    remainder='passthrough'
)
```

---

## Values Passed to `transformers`

Each tuple contains **3 values**:

```python
(name, transformer, columns)
```

### 1. `name`

* Any unique string
* Used to identify the transformation

Example:

```python
'imputer'
'encoder'
'scaler'
```

---

### 2. `transformer`

The preprocessing object to apply.

Examples:

```python
SimpleImputer()
OneHotEncoder()
OrdinalEncoder()
StandardScaler()
```

---

### 3. `columns`

List of columns on which the transformer will work.

Examples:

```python
['fever']
['gender', 'city']
['age', 'salary']
```

---

## Example Code

```python
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

transformer = ColumnTransformer(
    transformers=[

        ('imp',
         SimpleImputer(strategy='most_frequent'),
         ['fever']),

        ('ord',
         OrdinalEncoder(categories=[['Mild', 'Strong']]),
         ['cough']),

        ('ohe',
         OneHotEncoder(
             sparse_output=False,
             drop='first'
         ),
         ['gender', 'city'])

    ],
    remainder='passthrough'
)
```

---

## Meaning of Each Line

```python
('imp', SimpleImputer(), ['fever'])
```

* Name → `'imp'`
* Transformer → `SimpleImputer()`
* Column → `'fever'`

---

```python
('ord', OrdinalEncoder(), ['cough'])
```

* Name → `'ord'`
* Transformer → `OrdinalEncoder()`
* Column → `'cough'`

---

```python
('ohe', OneHotEncoder(), ['gender', 'city'])
```

* Name → `'ohe'`
* Transformer → `OneHotEncoder()`
* Columns → `'gender'`, `'city'`

---

## `remainder` Parameter

Controls columns not mentioned in `transformers`.

```python
remainder='passthrough'
```

Keep remaining columns unchanged.

```python
remainder='drop'
```

Remove remaining columns.

---

## Important Methods

```python
transformer.fit(X_train)

transformer.transform(X_test)

transformer.fit_transform(X_train)
```

---

## Rule to Remember

* Use `fit_transform()` on training data.
* Use `transform()` on test data.

Never use:

```python
transformer.fit_transform(X_test)
```

---

## Quick Formula

```text
ColumnTransformer =
[(name, transformer, columns)]
                ↓
        Apply preprocessing
        to selected columns
```
